<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/04_Analytics_Hub_Clean_Rooms_and_External_Datasets.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Farvind-dhariwal%2Faeon-credit-gcp-workshop%2Fmain%2Ftrack1_platform_governance%2Fnotebook%2F04_Analytics_Hub_Clean_Rooms_and_External_Datasets.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/notebook/04_Analytics_Hub_Clean_Rooms_and_External_Datasets.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/04_Analytics_Hub_Clean_Rooms_and_External_Datasets.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/04_Analytics_Hub_Clean_Rooms_and_External_Datasets.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

---

# Track 1 (Notebook 04 — Lab 1.4): Analytics Hub, BigQuery Data Clean Rooms & External Datasets
**AEON Credit Service Malaysia (ACSM) — Google Agentic Data Cloud Workshop (`asia-southeast1` Singapore)**

---

### 📋 Notebook 04 Step-by-Step Execution Summary
1. **Step 0 (Environment Setup & APIs)**: Auto-detect active `PROJECT_ID` and Singapore region (`asia-southeast1`), enable `analyticshub.googleapis.com`, and ensure workshop `.sql` and `.csv.gz` assets are staged.
2. **Step 1 (Invoke `.sql` Bootstrap Files)**: Ensure the 8 base tables exist in `acsm_bronze` (so this notebook can run independently without running Notebooks 01–03 first) and invoke [`05_cleanroom_and_external_datasets.sql`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/sql/05_cleanroom_and_external_datasets.sql) to initialize `acsm_cleanroom` and `acsm_subscribed_data`.
3. **Step 2 (Part A — Analytics Hub Zero-Copy Cross-Entity Data Sharing)**: Use `bq mk --data_exchange` and `bq ls --data_exchange` to create the **ACSM-AEON Ecosystem Exchange (`acsm_aeon_exchange`)** in `asia-southeast1` and query the curated **ACSM Merchant Spend Aggregations** listing with zero data duplication.
4. **Step 3 (Part B — BigQuery Data Clean Room: BNM RMiT & PDPA Compliant Collaboration)**: Create a Clean Room exchange via `bq mk --data_exchange` and execute a privacy-preserving join between **ACSM Cardholders (`m3CIF` + `Fact_CC_Sales`)** and **AEON Supermarket Loyalty Members (`acsm_cleanroom.partner_aeon_retail_shoppers`)** enforcing k-anonymity (`HAVING COUNT(DISTINCT c.CIF_ID) >= 20`).
5. **Step 4 (Part C.1 — Subscribing to Google Trends: Malaysian Financial Intent)**: Query regional Malaysian search interest for financing terms (`personal loan`, `credit card`, `car loan`, `installment plan`).
6. **Step 5 (Part C.2 — Subscribing to Malaysia Places Insight: Google Maps / Places POI)**: Enrich raw merchant spend locations (`Fact_CC_Sales.LDESC`) by joining with `acsm_subscribed_data.malaysia_places_catalog`.
7. **Step 6 (Part C.3 — Google Ads Data Integration: Card Acquisition Campaigns)**: Correlate digital marketing impressions & clicks (`acsm_subscribed_data.google_ads_campaign_performance`) with activated cards (`Card_Status = 'Active'`) in `acsm_bronze.dimProduct`.

---
## Step 0: Configure Parameters (`PROJECT_ID` & Singapore Region) and Enable Analytics Hub API
This cell auto-detects your active Google Cloud `PROJECT_ID`, sets `LOCATION = "asia-southeast1"` (Singapore), enables `analyticshub.googleapis.com` via `gcloud`, and clones the workshop repo if needed.

In [ ]:
# @title Step 0: Auto-Detect `PROJECT_ID`, Enable Analytics Hub API & Stage Source Data
import os
import subprocess
import google.auth

PROJECT_ID = ""  # @param {type:"string"}
LOCATION = "asia-southeast1"  # @param {type:"string"}
REPO_URL = "https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop.git"
REPO_DIR = "aeon-credit-gcp-workshop"

if not PROJECT_ID or PROJECT_ID == "<PROJECT_ID>":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()
if not PROJECT_ID:
    PROJECT_ID = subprocess.check_output(
        ["gcloud", "config", "get-value", "project"], text=True
    ).strip()
if not PROJECT_ID or PROJECT_ID == "(unset)":
    _, PROJECT_ID = google.auth.default()

BUCKET_NAME = f"acsm-workshop-landing-{PROJECT_ID}"
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION
os.environ["BUCKET_NAME"] = BUCKET_NAME

%load_ext google.cloud.bigquery

!gcloud config set project $PROJECT_ID
!gcloud services enable bigquery.googleapis.com analyticshub.googleapis.com storage.googleapis.com --project=$PROJECT_ID

![ -d aeon-credit-gcp-workshop ] || git clone --depth 1 $REPO_URL $REPO_DIR
!git -C $REPO_DIR pull --ff-only
!gcloud storage buckets describe gs://{BUCKET_NAME} >/dev/null 2>&1 || gcloud storage buckets create gs://{BUCKET_NAME} --location={LOCATION} --uniform-bucket-level-access
!gcloud storage cp aeon-credit-gcp-workshop/data/full_compressed/*.csv.gz gs://{BUCKET_NAME}/full_compressed/

print(f"✅ Active Project : {PROJECT_ID}")
print(f"✅ Active Region  : {LOCATION} (Singapore)")

---
## Step 1: Invoke `.sql` Bootstrap Files (`acsm_bronze`, `acsm_cleanroom` & `acsm_subscribed_data`)
If you run Notebook 04 standalone without running Notebooks 01–03 first, this step automatically ensures the 8 base tables exist in `acsm_bronze` (via `00_create_8_tables_ddl_with_descriptions.sql` and `01_load_data_from_gcs.sql`), and invokes [`05_cleanroom_and_external_datasets.sql`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/sql/05_cleanroom_and_external_datasets.sql) to create `acsm_cleanroom` and `acsm_subscribed_data` in `asia-southeast1`.

In [ ]:
# @title Step 1.1: Invoke `.sql` Files to Bootstrap `acsm_bronze`, `acsm_cleanroom` & `acsm_subscribed_data`
!bq show --project_id=$PROJECT_ID $PROJECT_ID:acsm_bronze.Fact_CC_Sales >/dev/null 2>&1 || ( \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/00_create_8_tables_ddl_with_descriptions.sql && \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/01_load_data_from_gcs.sql \
)
!bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/05_cleanroom_and_external_datasets.sql
print("✅ Executed 05_cleanroom_and_external_datasets.sql (`acsm_cleanroom` & `acsm_subscribed_data` ready)!")

---
## Step 2 (Lab 1.4 — Part A): Analytics Hub Zero-Copy Cross-Entity Data Sharing (`ACSM` $\rightarrow$ `AEON Retail`)
Demonstrate how ACSM shares curated credit and merchant performance data with **AEON Retail (supermarkets and department stores)** without copying files or managing SFTP transfers using the `bq` CLI:

In [ ]:
# @title Step 2.1: Create Analytics Hub Data Exchange & List Active Exchanges (`bq` CLI)
# Create an Analytics Hub Data Exchange in Singapore (asia-southeast1)
!bq mk --location=asia-southeast1 \
    --data_exchange \
    --display_name="ACSM_AEON_Ecosystem_Exchange" \
    $PROJECT_ID:acsm_aeon_exchange || true

# List active listings/exchanges in the project
!bq ls --location=asia-southeast1 --data_exchange $PROJECT_ID:acsm_aeon_exchange

### Step 2.2: Query Curated Listing — ACSM Merchant Spend Aggregations
Subscribers (AEON Retail) query this linked dataset live with **zero data duplication**:

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Publish Curated Listing: ACSM Merchant Spend Aggregations
-- Subscribers query this linked dataset live with zero data duplication:
SELECT 
  p.PriviledgeMerchantsGrp AS merchant_group,
  p.LDESC AS location_name,
  COUNT(1) AS total_transactions,
  ROUND(SUM(p.Amount), 2) AS total_spending_rm
FROM `acsm_bronze.Fact_CC_Sales` p
GROUP BY 1, 2
ORDER BY total_spending_rm DESC
LIMIT 15;

---
## Step 3 (Lab 1.4 — Part B): BigQuery Data Clean Room (Privacy-Preserving Partner Collaboration)
Demonstrate a **Bank Negara Malaysia (BNM) RMiT** and **Malaysian PDPA-compliant** Data Clean Room join between **ACSM Cardholders (`acsm_bronze.m3CIF` + `acsm_bronze.Fact_CC_Sales`)** and **AEON Supermarket Loyalty Members (`acsm_cleanroom.partner_aeon_retail_shoppers`)**, discovering overlapping customer spend without either party exposing raw PII.

First, create the Clean Room exchange and dataset in `asia-southeast1` using `bq` CLI commands:

In [ ]:
# @title Step 3.1: Create Data Clean Room Exchange & Verify Clean Room Tables (`bq` CLI)
!bq mk --location=asia-southeast1 \
    --data_exchange \
    --display_name="ACSM_AEON_Retail_Data_Clean_Room" \
    $PROJECT_ID:acsm_aeon_cleanroom_exchange || true

!bq ls --location=asia-southeast1 $PROJECT_ID:acsm_cleanroom

### Step 3.2: Data Clean Room Analysis Query with Privacy-Preserving Thresholds (`k >= 20`)
Enforces **k-anonymity (minimum 20 matching entities)** via `HAVING COUNT(DISTINCT c.CIF_ID) >= 20` and prevents single-record identification:

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Data Clean Room Analysis Query with Privacy-Preserving Thresholds
-- Enforces k-anonymity (minimum 20 matching entities) and prevents single-record identification:
SELECT 
  c.State,
  c.MaritalSts,
  COUNT(DISTINCT c.CIF_ID) AS matching_joint_customers,
  ROUND(AVG(c.B_GrossIncome), 2) AS avg_monthly_income_rm,
  ROUND(SUM(s.Amount), 2) AS total_card_spend_rm,
  ROUND(SUM(r.retail_spend_amt), 2) AS total_aeon_supermarket_spend_rm
FROM `acsm_bronze.m3CIF` c
JOIN `acsm_bronze.Fact_CC_Sales` s ON c.CIF_ID = s.CIF_No
JOIN `acsm_cleanroom.partner_aeon_retail_shoppers` r ON c.CIF_ID = r.hashed_cif_match
GROUP BY 1, 2
HAVING COUNT(DISTINCT c.CIF_ID) >= 20 -- Privacy enforcement threshold
ORDER BY total_card_spend_rm DESC;

---
## Step 4 (Lab 1.4 — Part C): Subscribing to External Google Datasets

### 1. Subscribing to Google Trends (Malaysian Financial Intent)
Enrich ACSM credit underwriting records with Malaysian search sentiment for financing terms (`personal loan`, `credit card`, `car loan`, `installment plan`):
* **Public Dataset Reference (`US` multi-region)**: `bigquery-public-data.google_trends.international_top_terms`
* **Subscribed Regional Dataset (`asia-southeast1`)**: `acsm_subscribed_data.google_trends_malaysia_financial_intent`

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Querying Subscribed Google Trends Dataset for Macro Financing Demand in Malaysia
SELECT 
  term,
  region_name AS malaysian_state,
  score AS search_interest_index,
  date
FROM `acsm_subscribed_data.google_trends_malaysia_financial_intent`
WHERE country_name = 'Malaysia'
  AND term IN ('personal loan', 'credit card', 'car loan', 'installment plan')
  AND date >= DATE_SUB(CURRENT_DATE(), INTERVAL 60 DAY)
ORDER BY search_interest_index DESC
LIMIT 15;

### 2. Subscribing to Malaysia Places Insight (Google Maps / Places POI)
Geospatially enrich raw merchant spend locations (`Fact_CC_Sales.LDESC`) by joining with Google Places data (`acsm_subscribed_data.malaysia_places_catalog`) to standardize store categories:

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Enriching ACSM Spend Records with Google Places POI Catalog
SELECT 
  s.CIF_No,
  s.LDESC AS raw_merchant_location,
  poi.place_name AS verified_mall_name,
  poi.business_category AS standardized_category,
  poi.state AS merchant_state,
  s.Amount AS spend_amount_rm
FROM `acsm_bronze.Fact_CC_Sales` s
JOIN `acsm_subscribed_data.malaysia_places_catalog` poi 
  ON LOWER(s.LDESC) LIKE CONCAT('%', LOWER(poi.location_keyword), '%')
LIMIT 10;

### 3. Google Ads Data Integration (Card Acquisition Campaigns)
Correlate digital marketing acquisition impressions with actual card activation records in `acsm_bronze.dimProduct`:

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Correlate Digital Marketing Acquisition Impressions with Card Activation Records in dimProduct
SELECT 
  a.campaign_name,
  a.ad_network_type,
  MAX(a.impressions) AS total_impressions,
  MAX(a.clicks) AS total_clicks,
  COUNT(DISTINCT p.Account_No) AS activated_card_conversions,
  ROUND(SUM(p.CP_CL_Usage), 2) AS total_first_month_usage_rm
FROM `acsm_subscribed_data.google_ads_campaign_performance` a
JOIN `acsm_bronze.dimProduct` p ON a.campaign_id = p.Wallet_Tier
WHERE p.Card_Status = 'Active'
GROUP BY 1, 2
ORDER BY total_first_month_usage_rm DESC;

---
## ✅ Track 1 (Notebook 04 — Lab 1.4) Summary
| Lab 1.4 Section | CLI / SQL Capability Demonstrated (`asia-southeast1`) | Business & Compliance Value |
| :--- | :--- | :--- |
| **Part A: Analytics Hub Zero-Copy Data Sharing** | `bq mk --data_exchange` (`acsm_aeon_exchange`) + Merchant Spend Aggregation SQL on `Fact_CC_Sales` | Shares curated ACSM merchant spend with **AEON Retail** live without file copies or SFTP pipelines. |
| **Part B: BigQuery Data Clean Room** | `bq mk --data_exchange` (`acsm_aeon_cleanroom_exchange`) + Privacy-Preserving Join (`HAVING COUNT(DISTINCT c.CIF_ID) >= 20`) | BNM RMiT & Malaysian PDPA-compliant joint customer analysis with AEON Supermarket Loyalty (`k >= 20` anonymity threshold). |
| **Part C.1: Google Trends** | `acsm_subscribed_data.google_trends_malaysia_financial_intent` | Regional Malaysian search interest for `personal loan`, `credit card`, `car loan`, and `installment plan`. |
| **Part C.2: Malaysia Places Insight** | `acsm_bronze.Fact_CC_Sales` $\bowtie$ `acsm_subscribed_data.malaysia_places_catalog` | Enriches raw merchant locations (`LDESC`) with verified mall names, standardized store categories, and states. |
| **Part C.3: Google Ads Integration** | `acsm_subscribed_data.google_ads_campaign_performance` $\bowtie$ `acsm_bronze.dimProduct` | Connects Google Ads impressions & clicks to activated credit cards (`Card_Status = 'Active'`) and first-month card spend (`CP_CL_Usage`). |